<a href="https://colab.research.google.com/github/Chalhotra/ViT-Token-Economy/blob/main/notebooks/01_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baselines: DeiT and ViT Family (ImageNet-100)

This notebook follows the same structure as the EViT/ToMe notebooks:
- Shared utility plotting functions
- Explicit per-model run cells
- Final multi-model comparison visualizations

Models covered in this baseline notebook:
- DeiT-Tiny, DeiT-Small, DeiT-Base
- ViT-Tiny, ViT-Small, ViT-Base

## Setup (Colab)

Run this cell to clone the repository. For private repos, you'll need a GitHub token with repo access.

In [ ]:
# If running in a fresh Colab runtime, uncomment these lines:
# !git clone https://github.com/Chalhotra/ViT-Token-Economy.git
# %cd ViT-Token-Economy

In [2]:
!git checkout test-branch

Branch 'test-branch' set up to track remote branch 'test-branch' from 'origin'.
Switched to a new branch 'test-branch'


In [3]:
!pip -q install -r requirements.txt
!pip -q install -e .

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vit-deit-baselines (pyproject.toml) ... done


In [4]:
from src.imagenet_mapping import build_imagenet100_to_1k_map
from src.models import ModelConfig, create_model, shrink_imagenet1k_head_to_imagenet100
from src.data import DataConfig, load_imagenet100_split, build_transform_for_model, apply_timm_preprocess, build_loader
from src.eval import evaluate_accuracy_latency_throughput, compute_gflops
from src.utils import get_device, num_params
import torch

In [5]:
device = get_device()
maps = build_imagenet100_to_1k_map()

In [ ]:
from typing import List, Dict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def compare_baseline_models_spacious(
    results_list: List[Dict],
    title: str = "Baseline Multi-Model Comparison",
):
    """Create a spacious 2x2 comparison plot for any subset of baseline model runs."""
    if not results_list:
        print("No baseline results to plot.")
        return

    df = pd.DataFrame(results_list).copy()
    if df.empty:
        print("No baseline results to plot.")
        return

    for col in ["acc1", "gflops", "latency_ms", "throughput", "params_m"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["acc1", "gflops", "latency_ms"])
    if df.empty:
        print("No numeric baseline results available to plot.")
        return

    df = df.sort_values("model")
    n = len(df)
    x = np.arange(n)
    xlabels = df["model"].tolist()

    plt.close("all")
    fig, axes = plt.subplots(2, 2, figsize=(24, 16), dpi=130, constrained_layout=False)
    fig.subplots_adjust(left=0.06, right=0.985, bottom=0.24, top=0.90, wspace=0.28, hspace=0.42)

    ax = axes[0, 0]
    ax.bar(x, df["acc1"], color="steelblue", alpha=0.9)
    ax.set_title("Top-1 Accuracy", fontsize=16, pad=16)
    ax.set_ylabel("Accuracy (%)", fontsize=13, labelpad=10)
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, rotation=45, ha="right", fontsize=10)
    ax.grid(True, axis="y", alpha=0.30)

    ax = axes[0, 1]
    ax.bar(x, df["gflops"], color="coral", alpha=0.9)
    ax.set_title("Computational Cost", fontsize=16, pad=16)
    ax.set_ylabel("GFLOPs", fontsize=13, labelpad=10)
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, rotation=45, ha="right", fontsize=10)
    ax.grid(True, axis="y", alpha=0.30)

    ax = axes[1, 0]
    ax.bar(x, df["latency_ms"], color="mediumseagreen", alpha=0.9)
    ax.set_title("Inference Latency", fontsize=16, pad=16)
    ax.set_ylabel("Latency (ms)", fontsize=13, labelpad=10)
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, rotation=45, ha="right", fontsize=10)
    ax.grid(True, axis="y", alpha=0.30)

    ax = axes[1, 1]
    ax.scatter(
        df["gflops"],
        df["acc1"],
        s=150,
        alpha=0.85,
        c=np.arange(n),
        cmap="viridis",
        edgecolors="black",
        linewidths=0.4,
    )
    ax.set_title("Efficiency: Accuracy vs GFLOPs", fontsize=16, pad=16)
    ax.set_xlabel("GFLOPs", fontsize=13, labelpad=10)
    ax.set_ylabel("Top-1 Accuracy (%)", fontsize=13, labelpad=10)
    ax.grid(True, alpha=0.30)

    for i, (_, row) in enumerate(df.iterrows()):
        xoff = 8 if i % 2 == 0 else -8
        yoff = 8 if i % 3 else -10
        ax.annotate(
            row["model"],
            (row["gflops"], row["acc1"]),
            xytext=(xoff, yoff),
            textcoords="offset points",
            fontsize=9,
            ha="left" if xoff > 0 else "right",
            va="bottom" if yoff > 0 else "top",
            bbox=dict(boxstyle="round,pad=0.22", facecolor="white", alpha=0.72, edgecolor="none"),
        )

    fig.suptitle(title, fontsize=20, y=0.965)
    plt.show()

    print("\n" + "=" * 80)
    print(f"{title} – Summary Table")
    print("=" * 80)
    print(df.to_string(index=False))
    print("=" * 80 + "\n")

## Utility Functions for Baseline Visualization

In [ ]:
def run_baseline_test(model_id: str, batch_size: int = 64):
    print(f"\n{'='*80}")
    print(f"Testing baseline model: {model_id}")
    print(f"{'='*80}\n")

    model = create_model(ModelConfig(model_id=model_id, pretrained=True))
    model = shrink_imagenet1k_head_to_imagenet100(model, maps.new_to_old_map, num_classes=100)
    model = model.to(device).eval()

    ds = load_imagenet100_split(DataConfig(split='validation'))
    transform = build_transform_for_model(model)
    ds_t = apply_timm_preprocess(ds, transform)
    loader = build_loader(ds_t, DataConfig(batch_size=batch_size, split='validation', shuffle=False))

    metrics = evaluate_accuracy_latency_throughput(model, loader, device)
    sample = ds_t[0]['pixel_values'].unsqueeze(0).to(device)
    gflops = compute_gflops(model, sample)

    result = {
        'model': model_id,
        'params_m': num_params(model) / 1e6,
        'gflops': gflops,
        **metrics,
    }

    print(f"Results for {model_id}:")
    print(f"  Top-1 Accuracy: {metrics['acc1']:.2f}%")
    print(f"  GFLOPs: {gflops:.3f}")
    print(f"  Latency: {metrics['latency_ms']:.2f} ms")
    print(f"  Throughput: {metrics['throughput']:.1f} samples/sec")

    return result

In [ ]:
# Baseline: DeiT-Tiny
if 'results' not in globals():
    results = []

name = 'deit_tiny_patch16_224'
results = [x for x in results if x.get('model') != name]
results.append(run_baseline_test(name))

In [ ]:
# Baseline: DeiT-Small
if 'results' not in globals():
    results = []

name = 'deit_small_patch16_224'
results = [x for x in results if x.get('model') != name]
results.append(run_baseline_test(name))

In [ ]:
# Baseline: DeiT-Base
if 'results' not in globals():
    results = []

name = 'deit_base_patch16_224'
results = [x for x in results if x.get('model') != name]
results.append(run_baseline_test(name))

Resolving data files: 100% 17/17 [00:00<00:00, 92963.71it/s]
Resolving data files: 100% 17/17 [00:00<00:00, 111237.39it/s]
Unsupported operator aten::add encountered 25 time(s)
Unsupported operator aten::scaled_dot_product_attention encountered 12 time(s)
Unsupported operator aten::gelu encountered 12 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
blocks.0.attn.attn_drop, blocks.1.attn.attn_drop, blocks.10.attn.attn_drop, blocks.11.attn.attn_drop, blocks.2.attn.attn_drop, blocks.3.attn.attn_drop, blocks.4.attn.attn_drop, blocks.5.attn.attn_drop, blocks.6.attn.attn_drop, blocks.7.attn.attn_drop, blocks.8.attn.attn_drop, blocks.9.attn.attn_drop
Model: vit_tiny_patch16_224
Params: 5.54 M
FLOPs:  1.08 GFLOPs (1x

In [ ]:
# Baseline: ViT-Tiny
if 'results' not in globals():
    results = []

name = 'vit_tiny_patch16_224'
results = [x for x in results if x.get('model') != name]
results.append(run_baseline_test(name))

model.safetensors: 100% 22.9M/22.9M [00:01<00:00, 13.1MB/s]  
Resolving data files: 100% 17/17 [00:00<00:00, 115564.29it/s]
Resolving data files: 100% 17/17 [00:00<00:00, 110035.75it/s]
Map: 100% 5000/5000 [01:23<00:00, 59.61 examples/s] 
Unsupported operator aten::add encountered 25 time(s)
Unsupported operator aten::scaled_dot_product_attention encountered 12 time(s)
Unsupported operator aten::gelu encountered 12 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
blocks.0.attn.attn_drop, blocks.1.attn.attn_drop, blocks.10.attn.attn_drop, blocks.11.attn.attn_drop, blocks.2.attn.attn_drop, blocks.3.attn.attn_drop, blocks.4.attn.attn_drop, blocks.5.attn.attn_drop, blocks.6.attn.attn_drop, blocks.7.attn.attn_drop,

In [ ]:
# Baseline: ViT-Small
if 'results' not in globals():
    results = []

name = 'vit_small_patch16_224'
results = [x for x in results if x.get('model') != name]
results.append(run_baseline_test(name))

Resolving data files: 100% 17/17 [00:00<00:00, 97408.70it/s]
Resolving data files: 100% 17/17 [00:00<00:00, 105791.05it/s]
Unsupported operator aten::add encountered 25 time(s)
Unsupported operator aten::scaled_dot_product_attention encountered 12 time(s)
Unsupported operator aten::gelu encountered 12 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
blocks.0.attn.attn_drop, blocks.1.attn.attn_drop, blocks.10.attn.attn_drop, blocks.11.attn.attn_drop, blocks.2.attn.attn_drop, blocks.3.attn.attn_drop, blocks.4.attn.attn_drop, blocks.5.attn.attn_drop, blocks.6.attn.attn_drop, blocks.7.attn.attn_drop, blocks.8.attn.attn_drop, blocks.9.attn.attn_drop
Model: deit_tiny_patch16_224
Params: 5.54 M
FLOPs:  1.08 GFLOPs (1

In [ ]:
# Baseline: ViT-Base
if 'results' not in globals():
    results = []

name = 'vit_base_patch16_224'
results = [x for x in results if x.get('model') != name]
results.append(run_baseline_test(name))

Resolving data files: 100% 17/17 [00:00<00:00, 85087.31it/s]
Resolving data files: 100% 17/17 [00:00<00:00, 133526.53it/s]
Unsupported operator aten::add encountered 25 time(s)
Unsupported operator aten::scaled_dot_product_attention encountered 9 time(s)
Unsupported operator aten::gelu encountered 12 time(s)
Unsupported operator aten::mul encountered 3 time(s)
Unsupported operator aten::softmax encountered 3 time(s)
Unsupported operator aten::mean encountered 3 time(s)
Unsupported operator aten::topk encountered 3 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
blocks.0.attn.attn_drop, blocks.1.attn.attn_drop, blocks.10.attn.attn_drop, blocks.11.attn.attn_drop, blocks.2.attn.attn_drop, blocks.4.attn.attn_drop

In [ ]:
# Final baseline visualization call (works with any subset of models already run)
if 'results' not in globals() or not results:
    print('No baseline results found. Run one or more baseline model cells first.')
else:
    compare_baseline_models_spacious(
        results,
        title=f'Baseline Comparison (Available Runs: {len(results)})'
    )